**Writing Viterbi Algorithm for the Primer**
<br>*Finding the log probability of a given path*


In [13]:
from decimal import Decimal, getcontext
import math

getcontext().prec = 50

def get_log_prob_of_a_given_path(state_path, sequence):
    assert len(state_path) == len(sequence), "State path and sequence must be of the same length."
    
    # Define transitions
    transitions = {
        'Start': {'E': Decimal('1.0')},
        'E': {'E': Decimal('0.9'), '5': Decimal('0.1')},
        '5': {'I': Decimal('1.0')},
        'I': {'I': Decimal('0.9')}
    }

    # Define emissions 
    emissions = {
        'E': {'A': Decimal('0.25'), 'C': Decimal('0.25'), 'G': Decimal('0.25'), 'T': Decimal('0.25')},
        '5': {'A': Decimal('0.05'), 'C': Decimal('0.0'), 'G': Decimal('0.95'), 'T': Decimal('0.0')},
        'I': {'A': Decimal('0.4'), 'C': Decimal('0.1'), 'G': Decimal('0.1'), 'T': Decimal('0.4')}
    }

    prob = Decimal('1.0')
    prev_state = 'Start'
    
    for i in range(len(sequence)):
        state = state_path[i]
        symbol = sequence[i]
        prob *= transitions[prev_state][state]
        prob *= emissions[state][symbol]
        prev_state = state
    if prev_state == 'I':
        prob*=Decimal('0.1')
    # Compute the natural log 
    log_prob = float(prob.ln())
    return prob, log_prob

state_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
sequence = "CTTCATGTGAAAGCAGACGTAAGTCA" 

prod, log_prob = get_log_prob_of_a_given_path(state_path, sequence)
print("Log probability =", round(log_prob,2))



Log probability = -41.22


*Using Viterbi Algorithm to find maximum likely path* 

In [14]:

getcontext().prec = 50

def viterbi(sequence):
    states = ['E', '5', 'I']
    
    transitions = {
        'Start': {'E': Decimal('1.0')},
        'E': {'E': Decimal('0.9'), '5': Decimal('0.1')},
        '5': {'I': Decimal('1.0')},
        'I': {'I': Decimal('0.9')}
    }
    
    emissions = {
        'E': {'A': Decimal('0.25'), 'C': Decimal('0.25'), 'G': Decimal('0.25'), 'T': Decimal('0.25')},
        '5': {'A': Decimal('0.05'), 'C': Decimal('0.0'), 'G': Decimal('0.95'), 'T': Decimal('0.0')},
        'I': {'A': Decimal('0.4'), 'C': Decimal('0.1'), 'G': Decimal('0.1'), 'T': Decimal('0.4')}
    }
    
    n = len(sequence)
    # V[t][state]: best log probability for a path ending in state at position t.
    V = [{} for _ in range(n)]
    # path[state] will record the best path (list of states) to that state at time t
    path = {}
    initial = 'E'
    first_obs = sequence[0]
    emiss_prob = emissions[initial].get(first_obs, Decimal('0'))
    if emiss_prob == 0:
        V[0][initial] = Decimal('-Infinity')
    else:
        V[0][initial] = transitions['Start'][initial].ln() + emiss_prob.ln()
    path[initial] = [initial]
    
    for s in states:
        if s != initial:
            V[0][s] = Decimal('-Infinity')
            path[s] = []

    for t in range(1, n):
        new_V = {}
        new_path = {}
        obs = sequence[t]
        for curr in states:
            best_prob = Decimal('-Infinity')
            best_prev = None
            for prev in states:
                if curr not in transitions.get(prev, {}):
                    continue
                trans_prob = transitions[prev][curr]
                emiss_prob = emissions[curr].get(obs, Decimal('0'))
                if emiss_prob == 0:
                    continue
                candidate = V[t-1][prev] + trans_prob.ln() + emiss_prob.ln()
                if candidate > best_prob:
                    best_prob = candidate
                    best_prev = prev
            new_V[curr] = best_prob
            if best_prev is not None and path[best_prev]:
                new_path[curr] = path[best_prev] + [curr]
            else:
                new_path[curr] = []
        V[t] = new_V
        path = new_path
    
    final_best_state = None
    final_log_prob = Decimal('-Infinity')
    for s in states:
        if s == 'I':
            candidate = V[n-1][s] + Decimal('0.1').ln()
        else:
            candidate = V[n-1][s]

        if candidate > final_log_prob:
            final_log_prob = candidate
            final_best_state = s
    best_path = path[final_best_state]
    
    return ''.join(best_path), float(final_log_prob)

# Primers Example:
sequence = "CTTCATGTGAAAGCAGACGTAAGTCA" 
best_state_path, log_probability = viterbi(sequence)
print("Best state path:", best_state_path)
print("Log probability:",round(log_probability,2))


Best state path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability: -38.68
